# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Released: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"License: {getattr(metadata, 'license', 'n/a')}")

## 2. Data Overview
Review available record sets (tables of data), fields (columns), and their `@id`s.

Let's inspect record sets and their fields provided in this dataset. All references are shown with their `@id`.

In [ ]:
# List all record sets by their @id, name and description
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):")
    for record_set in record_sets:
        print(f"  - @id: {record_set.id}")
        print(f"    name: {getattr(record_set, 'name', '(no name)')}")
        print(f"    description: {getattr(record_set, 'description', '(no description)')}")
        print(f"    Fields:")
        for field in record_set.fields:
            print(f"      - @id: {field.id}, name: {getattr(field, 'name', '(no name)')}")
        print()

# If no record set is declared, try to list file objects (for fallback exploration)
if not record_sets:
    print("Trying to list file objects as records (fallback exploration)...")
    file_objects = getattr(metadata, 'file_objects', [])
    for fobj in file_objects:
        print(f"FileObject @id: {fobj.id}, name: {getattr(fobj, 'name', '(no name)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. If no explicit record set is present, attempt to load any available records/set from the default dataset object.

In [ ]:
# --- Attempt to extract records from available record sets ---

# Helper: Choose the first record set if present, fallback to file object
chosen_record_set = None
if record_sets:
    chosen_record_set = record_sets[0]
    print(f"Using record set: @id={chosen_record_set.id}, name={getattr(chosen_record_set, 'name', chosen_record_set.id)}")
    records = list(dataset.records(record_set=chosen_record_set.id))
else:
    # fallback mode: Some datasets place records directly
    print("No explicit record sets detected. Trying to extract all records.")
    records = list(dataset.records())
    if records:
        print(f"Loaded {len(records)} records.")

# Load into DataFrame if there are records
if records:
    df = pd.DataFrame(records)
    print(f"Fields loaded: {df.columns.tolist()}")
    display(df.head())
else:
    print("No records could be loaded from the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In this cell, you can select a numeric field, filter records, normalize, and group data. All columns are referenced via their `@id` or column name as in the DataFrame.

In [ ]:
# Assumption: There is a numeric field such as 'cr:field/coverage_percent' (look in previous cells output for field ids)
import numpy as np

# Choose a numeric field by @id (adjust as available in dataset for your exploration)
if records:
    # Try to find a numeric column
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not numeric_fields:
        # Try to detect any columns that can be converted to float
        possible_numeric = []
        for col in df.columns:
            try:
                df[col].astype(float)
                possible_numeric.append(col)
            except Exception:
                continue
        if possible_numeric:
            numeric_field = possible_numeric[0]
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        else:
            numeric_field = df.columns[0]
    else:
        numeric_field = numeric_fields[0]

    print(f"Numeric field chosen for EDA: {numeric_field}")
    threshold = df[numeric_field].mean()
    # Filtering
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (mean):")
    display(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a non-numeric field
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No records available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a sample visualization cell for histogram of the selected numeric field and a boxplot grouped by the group field (if exists).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if records:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group field exists
    if group_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`.
- The Croissant schema was used to retrieve metadata, fields, and records programmatically, referencing all entities by their `@id`.
- We demonstrated data filtering, normalization, summary statistics, and basic visualization using field and record set IDs where available.
- Further scientific analysis can be conducted once fields and their biological interpretation are clarified, including deeper data processing or machine learning tasks.